In [1]:
import cv2
import os
from pathlib import Path

# --- Update paths ---
BASE_IMAGE = Path(r"E:\AutonomousVehicleProject\datasets\images\training\image_02")
BASE_LABEL = Path(r"E:\AutonomousVehicleProject\datasets\labels\training\label_02")
SEQ_ID = "0000"

IMAGE_SEQ = BASE_IMAGE / SEQ_ID
LABEL_FILE = BASE_LABEL / f"{SEQ_ID}.txt"
OUTPUT_VIDEO = Path(r"E:\AutonomousVehicleProject\outputs") / f"{SEQ_ID}_tracking.mp4"
OUTPUT_VIDEO.parent.mkdir(parents=True, exist_ok=True)

# --- Load labels ---
def load_labels(label_path):
    labels_by_frame = {}
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            frame = int(parts[0])
            track_id = int(parts[1])
            obj_type = parts[2]
            xmin, ymin, xmax, ymax = map(float, parts[6:10])
            labels_by_frame.setdefault(frame, []).append({
                "id": track_id,
                "type": obj_type,
                "bbox": [int(xmin), int(ymin), int(xmax), int(ymax)]
            })
    return labels_by_frame

labels_by_frame = load_labels(LABEL_FILE)
img_files = sorted(os.listdir(IMAGE_SEQ))

# --- Video writer setup ---
first_frame = cv2.imread(str(IMAGE_SEQ / img_files[0]))
height, width, _ = first_frame.shape
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, 10, (width, height))  # 10 FPS

# --- Loop through frames ---
for frame_idx, img_name in enumerate(img_files):
    img_path = IMAGE_SEQ / img_name
    img = cv2.imread(str(img_path))
    
    # Draw bounding boxes
    anns = labels_by_frame.get(frame_idx, [])
    for ann in anns:
        x1, y1, x2, y2 = ann["bbox"]
        tid = ann["id"]
        obj_type = ann["type"]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, f"{obj_type}:{tid}", (x1, max(15, y1-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # Write frame to video
    out.write(img)

out.release()
print(f"Video saved to {OUTPUT_VIDEO}")


Video saved to E:\AutonomousVehicleProject\outputs\0000_tracking.mp4


In [2]:
import numpy as np
import cv2
from pathlib import Path

# --- Preprocessing settings ---
TARGET_SIZE = (640, 480)  # width, height
OUTPUT_DIR = Path(r"E:\AutonomousVehicleProject\outputs")
OUTPUT_NPZ = OUTPUT_DIR / f"{SEQ_ID}_preprocessed.npz"
OUTPUT_VIDEO_PRE = OUTPUT_DIR / f"{SEQ_ID}_preprocessed.mp4"
FPS = 10

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Preprocessing functions ---
def preprocess_image(img, target_size=TARGET_SIZE):
    img_resized = cv2.resize(img, target_size)
    img_normalized = img_resized / 255.0
    return img_normalized

def preprocess_labels(anns, original_size, target_size=TARGET_SIZE):
    orig_h, orig_w = original_size
    target_w, target_h = target_size
    scale_x = target_w / orig_w
    scale_y = target_h / orig_h
    processed = []

    for ann in anns:
        x1, y1, x2, y2 = ann["bbox"]
        x1 = x1 * scale_x
        x2 = x2 * scale_x
        y1 = y1 * scale_y
        y2 = y2 * scale_y
        if x2 - x1 < 1: x2 = x1 + 1
        if y2 - y1 < 1: y2 = y1 + 1
        processed.append({
            "id": ann["id"],
            "type": ann["type"],
            "bbox": [int(x1), int(y1), int(x2), int(y2)]
        })
    return processed

# --- Video writer setup ---
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_pre = cv2.VideoWriter(str(OUTPUT_VIDEO_PRE), fourcc, FPS, TARGET_SIZE)

sequence_data = []

# --- Process frames ---
for frame_idx, img_name in enumerate(img_files):
    img_path = IMAGE_SEQ / img_name
    img = cv2.imread(str(img_path))
    orig_h, orig_w = img.shape[:2]

    # Keep all objects, no filtering
    anns = labels_by_frame.get(frame_idx, [])

    # preprocess image and labels
    img_proc = preprocess_image(img)
    anns_proc = preprocess_labels(anns, (orig_h, orig_w))

    # store for model
    sequence_data.append({
        "frame": frame_idx,
        "image": img_proc,
        "labels": anns_proc
    })

    # --- Write preprocessed video for verification ---
    frame_pre = (img_proc * 255).astype(np.uint8)
    for ann in anns_proc:
        x1, y1, x2, y2 = ann["bbox"]
        tid = ann["id"]
        obj_type = ann["type"]
        cv2.rectangle(frame_pre, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame_pre, f"{obj_type}:{tid}", (x1, max(15, y1-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    out_pre.write(frame_pre)

out_pre.release()

# --- Save preprocessed data ---
images = np.array([f["image"] for f in sequence_data], dtype=np.float32)
bboxes = np.array([f["labels"] for f in sequence_data], dtype=object)  # <- use object to handle variable number
np.savez_compressed(OUTPUT_NPZ, images=images, bboxes=bboxes)

print(f"Preprocessed video saved: {OUTPUT_VIDEO_PRE}")
print(f"Preprocessed data saved: {OUTPUT_NPZ}")


Preprocessed video saved: E:\AutonomousVehicleProject\outputs\0000_preprocessed.mp4
Preprocessed data saved: E:\AutonomousVehicleProject\outputs\0000_preprocessed.npz


In [3]:
import numpy as np
import os
from pathlib import Path

# --- Load preprocessed data ---
OUTPUT_DIR = Path(r"E:\AutonomousVehicleProject\outputs")
NPZ_FILE = OUTPUT_DIR / f"{SEQ_ID}_preprocessed.npz"
data = np.load(NPZ_FILE, allow_pickle=True)
images = data['images']      # shape: (num_frames, H, W, C)
bboxes = data['bboxes']      # list of dicts per frame

# --- Create YOLO dataset folders ---
YOLO_DIR = OUTPUT_DIR / "yolo_dataset"
IMG_DIR = YOLO_DIR / "images"
LBL_DIR = YOLO_DIR / "labels"

for d in [IMG_DIR, LBL_DIR]:
    (d / "train").mkdir(parents=True, exist_ok=True)
    (d / "val").mkdir(parents=True, exist_ok=True)

# --- Map object types to class IDs ---
all_types = set()
for frame in bboxes:
    for ann in frame:
        all_types.add(ann['type'])
class_map = {obj: idx for idx, obj in enumerate(sorted(all_types))}
print("Class map:", class_map)

# --- Split train/val (80/20) ---
num_frames = len(images)
split_idx = int(0.8 * num_frames)
train_indices = range(split_idx)
val_indices = range(split_idx, num_frames)

def save_yolo_files(indices, split):
    for i in indices:
        img = (images[i] * 255).astype(np.uint8)
        img_name = f"{i:05d}.png"
        label_name = f"{i:05d}.txt"

        # Save image
        cv2.imwrite(str(IMG_DIR / split / img_name), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

        # Save label
        h, w, _ = img.shape
        yolo_lines = []
        for ann in bboxes[i]:
            class_id = class_map[ann['type']]
            x1, y1, x2, y2 = ann['bbox']
            xc = ((x1 + x2) / 2) / w
            yc = ((y1 + y2) / 2) / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h
            yolo_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

        with open(LBL_DIR / split / label_name, 'w') as f:
            f.write("\n".join(yolo_lines))

save_yolo_files(train_indices, "train")
save_yolo_files(val_indices, "val")
print("YOLO dataset created!")


Class map: {'Car': 0, 'Cyclist': 1, 'DontCare': 2, 'Pedestrian': 3, 'Van': 4}
YOLO dataset created!


In [4]:
import yaml
yolo_yaml = {
    "train": str(IMG_DIR / "train"),
    "val": str(IMG_DIR / "val"),
    "nc": len(class_map),
    "names": [k for k,v in sorted(class_map.items(), key=lambda item: item[1])]
}
with open(OUTPUT_DIR / "yolo_dataset.yaml", "w") as f:
    yaml.dump(yolo_yaml, f)

from ultralytics import YOLO

# Load a pre-trained YOLOv8s model
model = YOLO("yolov8s.pt")  # small, fast baseline

# Train on your dataset
model.train(data=str(OUTPUT_DIR / "yolo_dataset.yaml"),
            epochs=50,
            imgsz=640,
            batch=8,
            project=str(OUTPUT_DIR / "yolo_training"),
            name="baseline")
results = model.predict(source=str(IMG_DIR / "val"), show=True, conf=0.5)


Ultralytics 8.3.203  Python-3.13.7 torch-2.8.0+cpu CPU (11th Gen Intel Core(TM) i3-1115G4 3.00GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\AutonomousVehicleProject\outputs\yolo_dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=baseline6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

In [5]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# --- Paths ---
OUTPUT_DIR = Path(r"E:\AutonomousVehicleProject\outputs")
IMG_DIR = OUTPUT_DIR / "yolo_dataset/images/val"
LBL_DIR = OUTPUT_DIR / "yolo_dataset/labels/val"
MODEL_PATH = OUTPUT_DIR / "yolo_training/baseline/weights/best.pt"
OUTPUT_VIDEO = OUTPUT_DIR / "baseline_inference_comparison.mp4"
FPS = 10

# --- Load model ---
model = YOLO(str(MODEL_PATH))

# --- Setup video writer ---
sample_img = cv2.imread(str(next(IMG_DIR.glob("*.png"))))
h, w, _ = sample_img.shape
out_vid = cv2.VideoWriter(str(OUTPUT_VIDEO),
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          FPS, (w*2, h))  # side-by-side

# --- Get class map ---
with open(OUTPUT_DIR / "yolo_dataset.yaml") as f:
    import yaml
    yolo_yaml = yaml.safe_load(f)
class_names = yolo_yaml["names"]

# --- Process each validation image ---
img_files = sorted(IMG_DIR.glob("*.png"))
for img_path in img_files:
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- Ground truth ---
    label_file = LBL_DIR / (img_path.stem + ".txt")
    gt_img = img.copy()
    if label_file.exists():
        with open(label_file) as f:
            for line in f:
                parts = line.strip().split()
                class_id, xc, yc, bw, bh = map(float, parts)
                class_id = int(class_id)
                # Convert normalized to absolute coordinates
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                cv2.rectangle(gt_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(gt_img, class_names[class_id], (x1, max(15, y1-5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    # --- Model prediction ---
    results = model.predict(img_path, conf=0.5, verbose=False)
    pred_img = img.copy()
    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            class_id = int(box.cls[0])
            cv2.rectangle(pred_img, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(pred_img, class_names[class_id], (x1, max(15, y1-5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    # --- Combine side-by-side ---
    combined = np.zeros((h, w*2, 3), dtype=np.uint8)
    combined[:, :w] = gt_img
    combined[:, w:] = pred_img

    # Write to video
    out_vid.write(combined)

out_vid.release()
print(f"Inference comparison video saved at: {OUTPUT_VIDEO}")


Inference comparison video saved at: E:\AutonomousVehicleProject\outputs\baseline_inference_comparison.mp4


In [6]:
import cv2
import os
from pathlib import Path
from deep_sort_realtime.deepsort_tracker import DeepSort
import numpy as np

# --- Paths ---
SEQ_ID = "0000"
IMAGE_SEQ = Path(r"E:\AutonomousVehicleProject\datasets\images\training\image_02") / SEQ_ID
LABEL_FILE = Path(r"E:\AutonomousVehicleProject\datasets\labels\training\label_02") / f"{SEQ_ID}.txt"
OUTPUT_DIR = Path(r"E:\AutonomousVehicleProject\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO_TRACK = OUTPUT_DIR / f"{SEQ_ID}_baseline_tracking.avi"
TRACKER_RESULTS_FILE = OUTPUT_DIR / f"{SEQ_ID}_baseline_tracking_results.txt"

# --- Tracker setup ---
tracker = DeepSort(max_age=30, n_init=3, nms_max_overlap=1.0)

# --- Video writer setup ---
first_frame = cv2.imread(str(IMAGE_SEQ / sorted(os.listdir(IMAGE_SEQ))[0]))
height, width, _ = first_frame.shape
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(str(OUTPUT_VIDEO_TRACK), fourcc, 10, (width, height))

# --- Store predictions here ---
pred_by_frame = {}

# --- Class colors for drawing ---
COLORS = {}
def get_color(cls):
    if cls not in COLORS:
        COLORS[cls] = tuple(np.random.randint(0, 255, 3).tolist())
    return COLORS[cls]

# --- Open tracker results file for writing ---
with open(TRACKER_RESULTS_FILE, "w") as f_out:

    # --- Loop through frames ---
    img_files = sorted(os.listdir(IMAGE_SEQ))
    for frame_idx, img_name in enumerate(img_files):
        img_path = IMAGE_SEQ / img_name
        frame = cv2.imread(str(img_path))

        # Ground-truth labels for class mapping
        anns = labels_by_frame.get(frame_idx, [])

        # Convert to DeepSORT input: (x1,y1,w,h,confidence,class)
        detections = []
        for ann in anns:
            x1, y1, x2, y2 = ann["bbox"]
            w, h = x2 - x1, y2 - y1
            detections.append(([x1, y1, w, h], 1.0, ann["type"]))

        # Update tracker
        tracks = tracker.update_tracks(detections, frame=frame)

        # Save predictions
        pred_by_frame[frame_idx] = []
        for t in tracks:
            if not t.is_confirmed():
                continue
            track_id = t.track_id
            ltrb = t.to_ltrb()  # [x1,y1,x2,y2]
            obj_class = t.get_det_class() if t.get_det_class() else "Unknown"

            pred_by_frame[frame_idx].append({
                "id": track_id,
                "bbox": [int(l) for l in ltrb],
                "class": obj_class
            })

            # Draw box
            x1, y1, x2, y2 = map(int, ltrb)
            color = get_color(obj_class)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"{obj_class}:{track_id}", (x1, max(15, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

            # --- Write to tracker results file (KITTI-like) ---
            # Format: frame track_id class left top right bottom
            f_out.write(f"{frame_idx} {track_id} {obj_class} {x1} {y1} {x2} {y2}\n")

        # Write frame to video
        out.write(frame)

out.release()
cv2.destroyAllWindows()
print(f"✅ Tracking video saved: {OUTPUT_VIDEO_TRACK}")
print(f"✅ Tracker results saved: {TRACKER_RESULTS_FILE}")


C:\Users\rtnee\AppData\Local\Programs\Python\Python313\Lib\site-packages\deep_sort_realtime\embedder\embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


✅ Tracking video saved: E:\AutonomousVehicleProject\outputs\0000_baseline_tracking.avi
✅ Tracker results saved: E:\AutonomousVehicleProject\outputs\0000_baseline_tracking_results.txt


In [19]:
import os
import numpy as np

# ---------------------------
# Robust KITTI Label Parser
# ---------------------------
def parse_kitti_labels(label_file):
    """
    Parse KITTI labels into a dictionary:
    { frame_id : [ (x1, y1, x2, y2, obj_type) ] }
    """
    labels = {}
    with open(label_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 15:
                continue  # skip malformed lines

            frame_id = int(parts[0])
            obj_type = parts[2]

            # skip DontCare regions
            if obj_type == "DontCare":
                continue

            # bounding box coordinates (floats)
            x1, y1, x2, y2 = map(float, parts[6:10])

            if frame_id not in labels:
                labels[frame_id] = []
            labels[frame_id].append((x1, y1, x2, y2, obj_type))

    return labels

# ---------------------------
# Load Labels & Print Sample
# ---------------------------
LABEL_FILE = r"E:\AutonomousVehicleProject\datasets\labels\training\label_02\0000.txt"

gt_labels = parse_kitti_labels(LABEL_FILE)

print(f"✅ Parsed {len(gt_labels)} frames with annotations.")
for frame_id in sorted(gt_labels.keys())[:3]:
    print(f"Frame {frame_id}: {gt_labels[frame_id]}")


✅ Parsed 154 frames with annotations.
Frame 0: [(296.744956, 161.752147, 455.226042, 292.372804, 'Van'), (737.619499, 161.531951, 931.112229, 374.0, 'Cyclist'), (1106.137292, 166.576807, 1204.470628, 323.876144, 'Pedestrian')]
Frame 1: [(294.898777, 156.024256, 452.199718, 284.621269, 'Van'), (745.017137, 156.393157, 938.839722, 374.0, 'Cyclist'), (1138.342096, 160.872449, 1223.338201, 324.146788, 'Pedestrian')]
Frame 2: [(293.09356, 150.470149, 449.259225, 277.10429, 'Van'), (752.406083, 151.248515, 946.56249, 374.0, 'Cyclist'), (1151.358043, 154.633575, 1223.691377, 324.375836, 'Pedestrian')]


In [20]:
import numpy as np

# ---------------------------
# Parse Tracker Results
# ---------------------------
def parse_tracker_results(results_file):
    """
    Parse tracker results into dict:
    { frame_id : [ (x1, y1, x2, y2, track_id, cls) ] }
    """
    tracker_data = {}
    with open(results_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            frame_id = int(parts[0])
            track_id = int(parts[1])
            cls = parts[2]
            x1, y1, x2, y2 = map(float, parts[3:7])
            if frame_id not in tracker_data:
                tracker_data[frame_id] = []
            tracker_data[frame_id].append((x1, y1, x2, y2, track_id, cls))
    return tracker_data

# ---------------------------
# IoU helper
# ---------------------------
def iou(bb1, bb2):
    x1, y1, x2, y2 = bb1
    x1g, y1g, x2g, y2g = bb2
    inter_x1, inter_y1 = max(x1, x1g), max(y1, y1g)
    inter_x2, inter_y2 = min(x2, x2g), min(y2, y2g)
    iw, ih = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
    inter = iw * ih
    union = (x2-x1)*(y2-y1) + (x2g-x1g)*(y2g-y1g) - inter
    return inter / union if union > 0 else 0

# ---------------------------
# Evaluate Tracking
# ---------------------------
def evaluate_mot(gt_labels, tracker_data, iou_thr=0.5):
    total_gt, total_pred, matched, id_switches, total_iou = 0, 0, 0, 0, 0
    prev_match = {}

    for frame_id in sorted(gt_labels.keys()):
        gts = gt_labels.get(frame_id, [])
        preds = tracker_data.get(frame_id, [])

        total_gt += len(gts)
        total_pred += len(preds)

        used_preds = set()
        for gt in gts:
            gt_box = gt[:4]  # (x1,y1,x2,y2)
            best_iou, best_pred_idx = 0, -1

            for j, pred in enumerate(preds):
                if j in used_preds: 
                    continue
                pred_box = pred[:4]
                iou_score = iou(gt_box, pred_box)
                if iou_score > best_iou:
                    best_iou, best_pred_idx = iou_score, j

            if best_iou >= iou_thr:
                matched += 1
                total_iou += best_iou
                used_preds.add(best_pred_idx)

                # Check ID switches
                pred_id = preds[best_pred_idx][4]
                if gt not in prev_match:
                    prev_match[gt] = pred_id
                else:
                    if prev_match[gt] != pred_id:
                        id_switches += 1
                        prev_match[gt] = pred_id

    mota = 1 - ( (total_gt - matched) + id_switches ) / total_gt if total_gt > 0 else 0
    motp = total_iou / matched if matched > 0 else 0
    precision = matched / total_pred if total_pred > 0 else 0
    recall = matched / total_gt if total_gt > 0 else 0

    print(f"✅ Total GT objects: {total_gt}")
    print(f"✅ Total Predictions: {total_pred}")
    print(f"✅ Total Matched: {matched}")
    print(f"✅ Total ID Switches: {id_switches}")
    print(f"🔹 MOTA: {mota:.4f}")
    print(f"🔹 MOTP: {motp:.4f}")
    print(f"🔹 Precision: {precision:.4f}")
    print(f"🔹 Recall: {recall:.4f}")

# ---------------------------
# Run Evaluation
# ---------------------------
TRACKER_RESULTS_FILE = r"E:\AutonomousVehicleProject\outputs\0000_baseline_tracking_results.txt"

tracker_data = parse_tracker_results(TRACKER_RESULTS_FILE)
evaluate_mot(gt_labels, tracker_data, iou_thr=0.5)


✅ Total GT objects: 711
✅ Total Predictions: 1494
✅ Total Matched: 668
✅ Total ID Switches: 0
🔹 MOTA: 0.9395
🔹 MOTP: 0.8706
🔹 Precision: 0.4471
🔹 Recall: 0.9395
